# 📈 Stock / Commodity Forecast

Questo notebook scarica una serie temporale da Yahoo Finance e produce previsioni con un **pipeline di model selection**:

1. **Train/Validation split** – le ultime *H* osservazioni sono usate come validation set
2. **Training & Evaluation** – ogni modello viene trainato sul train set e valutato sul validation
3. **Selezione** – il modello migliore (MAPE più basso) viene selezionato
4. **Retrain & Forecast** – il vincitore viene ritrainato su tutta la serie e produce la previsione finale

### Modelli disponibili
| Modello | Tipo |
|---|---|
| **Chronos** | Amazon, zero-shot transformer |
| **N-HiTS** | NeuralForecast, trained on-the-fly |
| **N-BEATS** | NeuralForecast, trained on-the-fly |

---
### Ticker di esempio
| Asset | Ticker |
|---|---|
| Apple | `AAPL` |
| S&P 500 | `^GSPC` |
| Oro | `GC=F` |
| Petrolio WTI | `CL=F` |
| Bitcoin | `BTC-USD` |
| EUR/USD | `EURUSD=X` |

In [1]:
import sys

sys.path.insert(0, "../src")

from stock_forecast.data import download_series, get_ticker_info, PRESET_TICKERS
from stock_forecast.models import run_pipeline, run_all, run_chronos, run_nhits, run_nbeats
from stock_forecast.plot import plot_forecast, print_summary, print_validation_table

## 1 · Configurazione

In [10]:
# ── Parametri principali ──────────────────────────────────────────────
TICKER = "CL=F"  # Yahoo Finance ticker   (es: AAPL, ^GSPC, GC=F, BTC-USD)
PERIOD = "max"  # storico scaricato       (6mo | 1y | 2y | 5y | 10y | max)
INTERVAL = "1d"  # granularità             (1d | 1wk | 1mo)
HORIZON = 15  # giorni / barre da prevedere

# ── Opzioni modelli ───────────────────────────────────────────────────
CHRONOS_SIZE = "large"  # tiny | mini | small | base | large
MAX_STEPS = 600  # iterazioni di training per N-HiTS e N-BEATS
MODELS = ["N-HiTS", "N-BEATS", "Chronos"]  # modelli da confrontare
SELECTION_METRIC = "mae"  # metrica per la selezione (mae | rmse | mape)

print(f"Ticker  : {TICKER}  |  Periodo: {PERIOD}  |  Orizzonte: {HORIZON} barre")
print(f"Modelli : {', '.join(MODELS)}")
print(f"Metrica : {SELECTION_METRIC}")

Ticker  : CL=F  |  Periodo: max  |  Orizzonte: 15 barre
Modelli : N-HiTS, N-BEATS, Chronos
Metrica : mae


## 2 · Download dati

In [11]:
info = get_ticker_info(TICKER)
print(f"\nAsset   : {info.get('name', TICKER)}")
print(f"Ticker  : {info['ticker']}")
print(f"Currency: {info.get('currency', '—')}")
print(f"Sector  : {info.get('sector', '—')}")

df = download_series(TICKER, period=PERIOD, interval=INTERVAL)
print(f"\nSerie scaricata: {len(df)} osservazioni")
print(f"Da {df['ds'].min().date()}  →  {df['ds'].max().date()}")
df.tail()


Asset   : Crude Oil Jul 26
Ticker  : CL=F
Currency: USD
Sector  : —

Serie scaricata: 6470 osservazioni
Da 2000-08-23  →  2026-06-01


Price,ds,Close,High,Low,Open,Volume,y
6465,2026-05-26,93.889999,94.699997,89.410004,93.879997,358867,93.889999
6466,2026-05-27,88.680000,93.690002,87.769997,93.389999,290948,88.680000
6467,2026-05-28,88.900002,92.519997,87.110001,89.110001,235145,88.900002
6468,2026-05-29,87.360001,89.019997,86.349998,88.550003,235145,87.360001
6469,2026-06-01,94.110001,94.379997,88.449997,88.500000,142567,94.110001


## 3 · Model Selection Pipeline

Il pipeline:
1. Split train / validation (ultime `HORIZON` osservazioni)
2. Train & evaluate ogni modello sul validation set
3. Seleziona il migliore per MAPE
4. Retrain sul dataset completo → previsione finale

In [12]:
val_results, best_forecast = run_pipeline(
    df,
    horizon=HORIZON,
    models=MODELS,
    chronos_size=CHRONOS_SIZE,
    max_steps=MAX_STEPS,
    selection_metric=SELECTION_METRIC,
)


from stock_forecast.models import select_best_model

best_name = select_best_model(val_results, metric=SELECTION_METRIC)
print_validation_table(val_results, best_name=best_name)

print_summary({best_forecast.model_name: best_forecast}, ticker_info=info)

Seed set to 1


╔═══════════════════════════════════════════════════════╗
║          MODEL SELECTION PIPELINE                    ║
╚═══════════════════════════════════════════════════════╝

  Series length : 6470 observations
  Horizon       : 15 steps
  Val set size  : 15 (last 15 obs)
  Metric        : mae


── Validating: N-HiTS ─────────────────────────────────
  🏋️  Training N-HiTS (max_steps=600) …


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MQLoss        │      3 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  2.6 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.6 M                                                                                            
Non-trainable params: 3                                                                                            
Total params: 2.6 M                                                                                                
Total estimated model params size (MB): 10                                                                         
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.


  🔮 Forecasting 15 steps …


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()


════════════════════════════════════════════════════════════
  Model                  MAE       RMSE       MAPE       
  ───────────────────────────────────────────────────────
  N-HiTS              7.9470     9.7133      8.47%
  N-BEATS             5.9147     7.2307      6.07% 🏆
  Chronos             6.0689     7.0560      6.16%
════════════════════════════════════════════════════════════


═══════════════════════════════════════════════════════
  Crude Oil Jul 26  (CL=F)
  Currency : USD
  Sector   : —
═══════════════════════════════════════════════════════

  ┌─ N-BEATS
  │  Horizon      : 15 steps
  │  Last price   : 94.1100
  │  Forecast end : 97.0049  (+3.08%)
  │  80% CI       : [93.0417 – 97.6772]
  │  Val MAPE     : 6.07%
  └────────────────────────────────────────



## 4 · Grafico interattivo

In [13]:
fig = plot_forecast(
    best_forecast,
    ticker=info.get("name", TICKER),
    last_n_days=180,
    show_volume=True,
)
fig.show()

## 5 · Salva grafico

In [14]:
import os

os.makedirs("../outputs", exist_ok=True)
safe_ticker = TICKER.replace("=", "_").replace("^", "")
out_path = f"../outputs/{safe_ticker}_forecast.html"
fig.write_html(out_path)
print(f"Grafico salvato in: {out_path}")

Grafico salvato in: ../outputs/CL_F_forecast.html
